# Week 8 EXERCISE: Multi-Agent Task Orchestrator

## Overview

Build a **multi-agent system** where specialized agents collaborate to complete complex tasks.

**Use Case:** An autonomous task orchestrator that breaks down complex requests into subtasks, assigns them to specialized agents, and synthesizes results.

## Key Learnings from Week 8

- **Day 1**: Modal.com deployment, SpecialistAgent
- **Day 2**: RAG with vector stores, FrontierAgent, EnsembleAgent
- **Day 3**: ScannerAgent, MessagingAgent
- **Day 4**: AutonomousPlanningAgent with tool calling
- **Day 5**: Complete Deal Agent Framework

## Architecture

```
+-------------------+
|  User Request     |
+-------------------+
         |
         v
+-------------------+
|  Planner Agent    |  <-- Breaks down task, assigns to specialists
+-------------------+
         |
    +----+----+----+
    |    |    |    |
    v    v    v    v
+------+ +------+ +------+ +------+
|Research| |Code | |Review| |Summary|
|Agent  | |Agent| |Agent | |Agent |
+------+ +------+ +------+ +------+
    |    |    |    |
    +----+----+----+
         |
         v
+-------------------+
|  Synthesizer      |  <-- Combines results
+-------------------+
         |
         v
+-------------------+
|  Final Response   |
+-------------------+
```

---

In [ ]:
# Imports

import os
import json
import logging
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Callable
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
# Configuration

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Model configuration for different agents
PLANNER_MODEL = "openai/gpt-4.1-mini"      # Good at planning
SPECIALIST_MODEL = "openai/gpt-4.1-mini"   # Good at specific tasks
SYNTHESIZER_MODEL = "openai/gpt-4.1-mini"  # Good at summarization

# Initialize client
api_key = os.getenv("OPENROUTER_API_KEY")
if api_key:
    client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=api_key)
    print("Using OpenRouter API")
else:
    client = OpenAI()
    PLANNER_MODEL = "gpt-4.1-mini"
    SPECIALIST_MODEL = "gpt-4.1-mini"
    SYNTHESIZER_MODEL = "gpt-4.1-mini"
    print("Using OpenAI API directly")

## Agent Base Class

All agents inherit from this base class.

In [ ]:
@dataclass
class AgentResult:
    """Result from an agent's execution."""
    agent_name: str
    task: str
    result: str
    success: bool = True
    metadata: Dict = field(default_factory=dict)


class BaseAgent:
    """Base class for all agents."""
    
    def __init__(self, name: str, model: str, system_prompt: str):
        self.name = name
        self.model = model
        self.system_prompt = system_prompt
    
    def execute(self, task: str) -> AgentResult:
        """Execute the agent's task."""
        try:
            logger.info(f"[{self.name}] Executing: {task[:50]}...")
            response = client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": self.system_prompt},
                    {"role": "user", "content": task}
                ],
                temperature=0.7,
                max_tokens=1000
            )
            if response.choices:
                result = response.choices[0].message.content or ""
                logger.info(f"[{self.name}] Completed successfully")
                return AgentResult(
                    agent_name=self.name,
                    task=task,
                    result=result,
                    success=True
                )
            return AgentResult(
                agent_name=self.name,
                task=task,
                result="No response from model",
                success=False
            )
        except Exception as e:
            logger.error(f"[{self.name}] Error: {e}")
            return AgentResult(
                agent_name=self.name,
                task=task,
                result=f"Error: {str(e)}",
                success=False
            )

## Specialized Agents

Each agent has a specific role in the system.

In [ ]:
class ResearchAgent(BaseAgent):
    """Agent specialized in research and information gathering."""
    
    def __init__(self):
        super().__init__(
            name="Research Agent",
            model=SPECIALIST_MODEL,
            system_prompt="""You are a research specialist. Your role is to:
- Gather relevant information about topics
- Identify key facts, statistics, and insights
- Provide well-organized research summaries
- Cite sources when possible (even if simulated)

Be thorough but concise. Focus on accuracy and relevance."""
        )


class CodeAgent(BaseAgent):
    """Agent specialized in code generation and analysis."""
    
    def __init__(self):
        super().__init__(
            name="Code Agent",
            model=SPECIALIST_MODEL,
            system_prompt="""You are a coding specialist. Your role is to:
- Write clean, efficient code
- Explain code functionality
- Debug and fix issues
- Suggest best practices

Always include code comments and explain your approach."""
        )


class ReviewAgent(BaseAgent):
    """Agent specialized in reviewing and critiquing."""
    
    def __init__(self):
        super().__init__(
            name="Review Agent",
            model=SPECIALIST_MODEL,
            system_prompt="""You are a critical review specialist. Your role is to:
- Evaluate quality and accuracy of content
- Identify strengths and weaknesses
- Suggest improvements
- Provide constructive feedback

Be fair, thorough, and constructive in your reviews."""
        )


class WriterAgent(BaseAgent):
    """Agent specialized in writing and content creation."""
    
    def __init__(self):
        super().__init__(
            name="Writer Agent",
            model=SPECIALIST_MODEL,
            system_prompt="""You are a writing specialist. Your role is to:
- Create clear, engaging content
- Adapt tone and style to the audience
- Structure information logically
- Edit and polish text

Focus on clarity, engagement, and proper structure."""
        )

## Planner Agent with Tool Calling

The Planner Agent breaks down tasks and delegates to specialists.

In [ ]:
@dataclass
class TaskPlan:
    """A plan for completing a complex task."""
    original_request: str
    subtasks: List[Dict]
    strategy: str


class PlannerAgent:
    """Agent that plans and orchestrates other agents."""
    
    AVAILABLE_AGENTS = {
        "research": "For gathering information, facts, and context",
        "code": "For writing, analyzing, or debugging code",
        "review": "For critiquing, evaluating, and suggesting improvements",
        "writer": "For creating, editing, or polishing written content"
    }
    
    def __init__(self):
        self.name = "Planner Agent"
        self.model = PLANNER_MODEL
    
    def create_plan(self, request: str) -> TaskPlan:
        """Create a plan for the given request."""
        agents_desc = "\n".join([f"- {k}: {v}" for k, v in self.AVAILABLE_AGENTS.items()])
        
        prompt = f"""Analyze this request and create a plan to complete it using available agents.

REQUEST: {request}

AVAILABLE AGENTS:
{agents_desc}

Create a JSON plan with this structure:
{{
    "strategy": "Brief description of overall approach",
    "subtasks": [
        {{
            "agent": "agent_name",
            "task": "Specific task description",
            "order": 1
        }}
    ]
}}

Rules:
- Use 2-4 subtasks maximum
- Each subtask should be specific and actionable
- Order subtasks logically (some may run in parallel)
- Only use agents from the available list

Respond with ONLY the JSON, no other text."""
        
        try:
            response = client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=500
            )
            if response.choices:
                raw = response.choices[0].message.content or "{}"
                # Clean markdown code blocks if present
                raw = raw.strip()
                if raw.startswith("```"):
                    raw = raw.split("```")[1]
                    if raw.startswith("json"):
                        raw = raw[4:]
                raw = raw.strip()
                
                plan_data = json.loads(raw)
                return TaskPlan(
                    original_request=request,
                    subtasks=plan_data.get("subtasks", []),
                    strategy=plan_data.get("strategy", "Execute tasks sequentially")
                )
        except Exception as e:
            logger.error(f"Planning error: {e}")
        
        # Fallback plan
        return TaskPlan(
            original_request=request,
            subtasks=[{"agent": "research", "task": request, "order": 1}],
            strategy="Fallback: Single agent execution"
        )

## Synthesizer Agent

Combines results from multiple agents into a coherent response.

In [ ]:
class SynthesizerAgent:
    """Agent that synthesizes results from multiple agents."""
    
    def __init__(self):
        self.name = "Synthesizer Agent"
        self.model = SYNTHESIZER_MODEL
    
    def synthesize(self, original_request: str, results: List[AgentResult]) -> str:
        """Synthesize multiple agent results into a final response."""
        
        results_text = "\n\n".join([
            f"=== {r.agent_name} ===\nTask: {r.task}\nResult:\n{r.result}"
            for r in results
        ])
        
        prompt = f"""You are synthesizing results from multiple specialized agents to answer the user's request.

ORIGINAL REQUEST:
{original_request}

AGENT RESULTS:
{results_text}

Create a comprehensive, well-organized response that:
1. Directly addresses the original request
2. Integrates insights from all agents
3. Maintains a coherent narrative
4. Highlights key findings and recommendations

Do not mention the agents by name. Present the information as a unified response."""
        
        try:
            response = client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.5,
                max_tokens=1500
            )
            if response.choices:
                return response.choices[0].message.content or "Synthesis failed"
        except Exception as e:
            logger.error(f"Synthesis error: {e}")
        
        return "\n\n".join([r.result for r in results])

## Task Orchestrator

Coordinates the entire multi-agent workflow.

In [ ]:
class TaskOrchestrator:
    """Orchestrates multi-agent task execution."""
    
    def __init__(self):
        self.planner = PlannerAgent()
        self.synthesizer = SynthesizerAgent()
        self.agents = {
            "research": ResearchAgent(),
            "code": CodeAgent(),
            "review": ReviewAgent(),
            "writer": WriterAgent()
        }
        self.execution_log: List[str] = []
    
    def log(self, message: str):
        """Add to execution log."""
        self.execution_log.append(message)
        logger.info(message)
    
    def execute(self, request: str) -> Dict:
        """Execute a complex task using multiple agents."""
        self.execution_log = []
        
        # Step 1: Create plan
        self.log(f"Received request: {request[:100]}...")
        self.log("Creating execution plan...")
        plan = self.planner.create_plan(request)
        self.log(f"Strategy: {plan.strategy}")
        self.log(f"Subtasks: {len(plan.subtasks)}")
        
        # Step 2: Execute subtasks
        results: List[AgentResult] = []
        for subtask in sorted(plan.subtasks, key=lambda x: x.get("order", 0)):
            agent_name = subtask.get("agent", "research")
            task = subtask.get("task", request)
            
            if agent_name in self.agents:
                self.log(f"Executing {agent_name} agent: {task[:50]}...")
                agent = self.agents[agent_name]
                result = agent.execute(task)
                results.append(result)
                self.log(f"{agent_name} agent completed: {'Success' if result.success else 'Failed'}")
            else:
                self.log(f"Unknown agent: {agent_name}, skipping...")
        
        # Step 3: Synthesize results
        self.log("Synthesizing results...")
        final_response = self.synthesizer.synthesize(request, results)
        self.log("Synthesis complete!")
        
        return {
            "request": request,
            "plan": {
                "strategy": plan.strategy,
                "subtasks": plan.subtasks
            },
            "agent_results": [
                {
                    "agent": r.agent_name,
                    "task": r.task,
                    "result": r.result[:500] + "..." if len(r.result) > 500 else r.result,
                    "success": r.success
                }
                for r in results
            ],
            "final_response": final_response,
            "execution_log": self.execution_log
        }

## Test the Orchestrator

In [ ]:
# Create orchestrator
orchestrator = TaskOrchestrator()

# Test with a complex request
test_request = """I need help creating a Python function that calculates the Fibonacci sequence.
Please research best practices, write the code, and review it for improvements."""

result = orchestrator.execute(test_request)

print("=== EXECUTION LOG ===")
for log in result["execution_log"]:
    print(f"  {log}")

print("\n=== PLAN ===")
print(f"Strategy: {result['plan']['strategy']}")
for subtask in result['plan']['subtasks']:
    print(f"  [{subtask.get('order', 0)}] {subtask.get('agent')}: {subtask.get('task')[:60]}...")

print("\n=== FINAL RESPONSE ===")
print(result["final_response"])

## Gradio UI: Interactive Task Orchestrator

In [ ]:
# Global orchestrator instance
orchestrator = TaskOrchestrator()

def process_request(request: str) -> tuple:
    """Process a request and return formatted results."""
    if not request.strip():
        return "Please enter a request.", "", "", ""
    
    result = orchestrator.execute(request)
    
    # Format plan
    plan_md = f"""### Strategy
{result['plan']['strategy']}

### Subtasks
| Order | Agent | Task |
|-------|-------|------|
"""
    for subtask in result['plan']['subtasks']:
        plan_md += f"| {subtask.get('order', 0)} | {subtask.get('agent')} | {subtask.get('task')[:50]}... |\n"
    
    # Format agent results
    agents_md = ""
    for ar in result['agent_results']:
        status = "Success" if ar['success'] else "Failed"
        agents_md += f"""### {ar['agent']} ({status})
**Task:** {ar['task']}

{ar['result']}

---

"""
    
    # Format log
    log_text = "\n".join([f"[{i+1}] {log}" for i, log in enumerate(result['execution_log'])])
    
    return result['final_response'], plan_md, agents_md, log_text


# Example requests
EXAMPLE_REQUESTS = [
    "Write a Python function to check if a number is prime, with tests and documentation.",
    "Explain the concept of recursion in programming with examples and best practices.",
    "Create a simple REST API design for a todo list application.",
    "Research machine learning basics and write a beginner-friendly summary.",
    "Write a Python script to parse JSON files and review it for error handling."
]

def load_example(idx: int) -> str:
    """Load an example request."""
    if 0 <= idx < len(EXAMPLE_REQUESTS):
        return EXAMPLE_REQUESTS[idx]
    return ""

In [ ]:
# Build Gradio Interface
with gr.Blocks(title="Multi-Agent Task Orchestrator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""# Multi-Agent Task Orchestrator
    
Enter a complex task and watch specialized agents collaborate to complete it.

**Available Agents:**
- **Research Agent**: Gathers information and context
- **Code Agent**: Writes and analyzes code
- **Review Agent**: Evaluates and suggests improvements
- **Writer Agent**: Creates and polishes content
""")
    
    with gr.Row():
        with gr.Column(scale=2):
            request_input = gr.Textbox(
                label="Your Request",
                placeholder="Describe what you need help with...",
                lines=4
            )
            with gr.Row():
                submit_btn = gr.Button("Execute", variant="primary")
                clear_btn = gr.Button("Clear")
            
            gr.Markdown("### Example Requests:")
            with gr.Row():
                for i, ex in enumerate(EXAMPLE_REQUESTS[:3]):
                    btn = gr.Button(f"Example {i+1}", size="sm")
                    btn.click(fn=lambda idx=i: EXAMPLE_REQUESTS[idx], outputs=request_input)
    
    with gr.Tabs():
        with gr.Tab("Final Response"):
            response_output = gr.Markdown(label="Response")
        
        with gr.Tab("Execution Plan"):
            plan_output = gr.Markdown(label="Plan")
        
        with gr.Tab("Agent Results"):
            agents_output = gr.Markdown(label="Agent Results")
        
        with gr.Tab("Execution Log"):
            log_output = gr.Textbox(label="Log", lines=15, interactive=False)
    
    submit_btn.click(
        fn=process_request,
        inputs=request_input,
        outputs=[response_output, plan_output, agents_output, log_output]
    )
    clear_btn.click(
        fn=lambda: ("", "", "", "", ""),
        outputs=[request_input, response_output, plan_output, agents_output, log_output]
    )

# Launch
demo.launch()

## Advanced: Tool-Calling Planner

A more sophisticated planner that uses OpenAI's function calling.

In [ ]:
# Define tools for the planner
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "assign_research_task",
            "description": "Assign a research task to gather information",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string", "description": "The topic to research"}
                },
                "required": ["topic"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "assign_code_task",
            "description": "Assign a coding task to write or analyze code",
            "parameters": {
                "type": "object",
                "properties": {
                    "task": {"type": "string", "description": "The coding task description"}
                },
                "required": ["task"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "assign_review_task",
            "description": "Assign a review task to evaluate content",
            "parameters": {
                "type": "object",
                "properties": {
                    "content": {"type": "string", "description": "What to review"}
                },
                "required": ["content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "assign_writing_task",
            "description": "Assign a writing task to create content",
            "parameters": {
                "type": "object",
                "properties": {
                    "task": {"type": "string", "description": "The writing task description"}
                },
                "required": ["task"]
            }
        }
    }
]

def handle_tool_calls(tool_calls) -> List[Dict]:
    """Handle tool calls from the planner."""
    tasks = []
    agent_map = {
        "assign_research_task": "research",
        "assign_code_task": "code",
        "assign_review_task": "review",
        "assign_writing_task": "writer"
    }
    
    for i, call in enumerate(tool_calls):
        func_name = call.function.name
        args = json.loads(call.function.arguments)
        task_desc = args.get("topic") or args.get("task") or args.get("content", "")
        
        if func_name in agent_map:
            tasks.append({
                "agent": agent_map[func_name],
                "task": task_desc,
                "order": i + 1
            })
    
    return tasks

print("Tool-calling planner defined. Tools available:")
for tool in TOOLS:
    print(f"  - {tool['function']['name']}: {tool['function']['description']}")

## Summary

This exercise demonstrated:

1. **Agent Architecture**: Base class with specialized implementations
2. **Planning**: Breaking complex tasks into subtasks
3. **Tool Calling**: OpenAI function calling for agent assignment
4. **Orchestration**: Coordinating multiple agents
5. **Synthesis**: Combining results into coherent responses
6. **Gradio UI**: Interactive interface for the system

### Key Concepts from Week 8

| Concept | Implementation |
|---------|---------------|
| Specialist Agents | ResearchAgent, CodeAgent, ReviewAgent, WriterAgent |
| Planning Agent | PlannerAgent with JSON plan generation |
| Tool Calling | TOOLS definition for function calling |
| Orchestration | TaskOrchestrator coordinates workflow |
| Synthesis | SynthesizerAgent combines results |

### Potential Extensions

1. **Modal Deployment**: Deploy agents as serverless functions
2. **RAG Integration**: Add vector store for context retrieval
3. **Parallel Execution**: Run independent agents concurrently
4. **Memory**: Persist conversation history across sessions
5. **Streaming**: Stream agent responses in real-time